# Phase4: どの拡張が効く？ 写真が少なすぎたら？

> ⏱ 所要時間: 説明を読む 10分 ＋ 実行 約15分（`QUICK = True`）。時間がなければ、先に「1. アブレーション」だけ実行してください。

## このNotebookのゴール
1. **拡張を1種類ずつ試して、「効く拡張」「効かない拡張」を見分ける方法**を身に付ける
2. **学習用の写真の枚数を減らしたとき**、拡張の効果がどう変わるかを知る

## 前のNotebookとのつながり
Notebook 03 で、3つの拡張を組み合わせた `basic` が効くことが分かりました。
でも、**3つのうちどれが効いたのか？** もしかすると、**1つは無駄**かもしれません。
それを調べるのが今回の**アブレーション**です。

### アブレーション（切り分け実験）とは
**部品を1つずつ外したり、1つずつ試したりして、どの部品が効いているかを調べる実験**です。
全部入りで良くなっても、「どれが効いたか」「無駄なものが混ざっていないか」は分かりません。

## このNotebookでやること
- **1. アブレーション**: 5つの条件（拡張なし／反転のみ／切り抜きのみ／色調のみ／全部入り）を比べる
- **2. 学習枚数の影響**: クラスあたり10・30・60枚に減らして、拡張あり/なしを比べる

In [ ]:
import sys
from pathlib import Path

# 正しい環境（プロジェクトの .venv）で動いているか確認する
try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        f"torch が見つかりません。今のPython: {sys.executable}\n"
        "→ カーネルを「Python (image_augment_lab)」（または プロジェクトの .venv/bin/python）に切り替えてください。"
    )

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

## 設定

In [ ]:
QUICK = True                          # True: 各条件1回 / False: 3回の平均
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = ep(150)

## 1. アブレーション: 拡張を1種類ずつ

次の5つの条件を、**同じ設定**（150エポック）で比べます。

| 名前 | 内容 | 狙い |
|---|---|---|
| `none` | 拡張なし | 基準 |
| `flip` | 水平反転**のみ** | 反転だけの効果 |
| `crop` | ランダム切り抜き**のみ**（元の面積の60〜100%を切り出す） | 切り抜きだけの効果 |
| `color` | 色調変更**のみ** | 色調だけの効果 |
| `basic` | 切り抜き＋反転＋色調（**全部入り**） | Notebook 03 の組み合わせ |

**なぜ全部入りだけではダメなのか？** もし `color` が無駄でも、全部入りでは気付けません。
1つずつ試すことで、**「入れても無駄な拡張」を見つけて外せます**。

次のセルは5条件を学習します（`QUICK = True` で約7分）。**完了まで待ってください。**

In [ ]:
abl = []
for aug in ["none", "flip", "crop", "color", "basic"]:
    print(f"== aug={aug} ==")
    d, _ = run_seeds(SEEDS, aug=aug, epochs=EPOCHS)
    abl.append(d)
abl = pd.concat(abl)
mean_table(abl)

**表の見方**: `test_acc` が高いほど、初めて見る写真に強い。`gap` は小さいほど過学習が少ない。
次のセルは、`test_acc` を**棒グラフ**にします。棒の上の細い線（ヒゲ）は**ばらつき**（標準偏差）で、**短いほど安定**です。

In [ ]:
g = abl.groupby("aug", sort=False)["test_acc"]
order = ["none", "flip", "crop", "color", "basic"]
m, s = g.mean().loc[order], g.std().fillna(0).loc[order]   # 平均とばらつき
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(order, m, yerr=s, capsize=4, color=["gray", "C0", "C1", "C2", "C3"])
ax.set(ylim=(0.5, 1.0), ylabel="test精度", title="拡張の種類ごとの test 精度（平均±標準偏差）")
ax.grid(axis="y", alpha=0.3);

### 結果の読み方（MANUAL.mdの記載値・3シード平均、test精度）

| 条件 | test |
|---|---|
| 拡張なし | 0.701 |
| 水平反転のみ | 0.757 |
| **ランダム切り抜きのみ** | **0.835** |
| 色調変更のみ | 0.701 |
| 全部入り（basic） | 0.796 |

### 何が分かるか
1. **最も効いたのは、ランダム切り抜き**（拡張なしから +13 ポイント）。
   理由: 写真ごとに、アリの**位置や大きさが違う**。切り抜きは、それをAIに体験させるので、効果が大きい。
2. **水平反転も効いた**（+5.6 ポイント）。左右どちらを向いていてもアリと分かるようになる。
3. **色調変更だけでは、効果なし**（0.701のまま）。
   理由の推測: アリとハチを見分ける手がかりは主に**形**で、**色の変化は判別に役立たなかった**。
4. **全部入りより、切り抜き単独のほうが高い**（0.796 vs 0.835）。
   ただし標準偏差が約0.04あるので、**誤差の範囲**かもしれない。それでも、**「全部入りが最良」とは限らない**ことは言える。

> 💡 **この実験で得られる最大の教訓**: 拡張は**課題によって効く・効かないが違う**。1種類ずつ試さないと、無駄な拡張（今回は色調）に気付けない。

## 2. 学習用の写真の枚数を変えてみる

ここまでは243枚（アリ122＋ハチ121）で学習しました。**もっと少なくしたら、拡張の効果は大きくなる**でしょうか？
（直感的には、「少ないほど拡張が助けになる」と思えます。実際はどうでしょう？）

**クラスあたり10枚・30枚・60枚**を無作為に選んで学習し、**拡張なし vs 拡張あり（basic）**を比べます。
（`n_per_class=10` は「クラスあたり10枚」の意味。アリ10枚＋ハチ10枚＝合計20枚）
学習回数は少ない枚数でも足りるよう、**300エポック**にしています。

In [ ]:
sizes = []
for n in [10, 30, 60]:
    for aug in ["none", "basic"]:
        print(f"== クラスあたり{n}枚, aug={aug} ==")
        d, _ = run_seeds(SEEDS, aug=aug, epochs=ep(300), n_per_class=n)
        sizes.append(d)
sizes = pd.concat(sizes)
mean_table(sizes, by=("n_per_class", "aug"))

In [ ]:
# 枚数ごとの test 精度（左）と gap（右）を、拡張あり/なしで比べるグラフ
pv = sizes.assign(n=sizes.n_per_class.astype(int)).groupby(["n", "aug"])[["test_acc", "gap"]].mean().unstack()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
pv["test_acc"].plot(marker="o", ax=ax[0], title="test精度（大きいほど良い）")
pv["gap"].plot(marker="o", ax=ax[1], title="過学習の差 gap（小さいほど良い）")
for a in ax:
    a.set_xlabel("クラスあたり学習枚数"); a.grid(alpha=0.3)
fig.tight_layout();

### 結果の読み方（MANUAL.mdの記載値・3シード平均）

| クラスあたり枚数 | test（拡張なし → あり） | gap（拡張なし → あり） |
|---|---|---|
| 10 | 0.640 → 0.636 | 0.386 → 0.335 |
| 30 | 0.731 → 0.727 | 0.350 → 0.241 |
| 60 | 0.658 → 0.731 | 0.303 → 0.154 |

### 何が分かるか
1. **gap（過学習）は、どの枚数でも拡張で小さくなる。** 拡張は過学習を抑える。
2. **しかし、test の精度が上がったのは60枚のときだけ。** 10枚・30枚では**ほとんど変わらない**。
3. つまり、**拡張は「データの代わり」にはならない。**

> 🍎 たとえ話: 参考書が**1ページしかない**生徒に、その1ページの文字を大きくしたり、色を変えたりして見せても、**新しい知識は増えません**。
> 拡張は「**今ある情報の見え方を変える**」ことしかできず、**新しい情報は作れません**。

極端に写真が少ないときは、**写真を増やす**か、**転移学習**（次のNotebook）に頼る必要があります。

⚠ 注意: test は約77枚でばらつきが大きい（標準偏差0.04〜0.10）ため、これは**傾向の確認**にとどまります。

## 🧪 試してみよう

### 演習1: 切り抜きの強さを変えてみる
**やること**: `src/train.py` を開き、`AUG_SETS` の `"crop"` の **`scale=(0.6, 1.0)`** を **`scale=(0.2, 1.0)`** に書き換えて保存します。
**カーネルを再起動**して、このNotebookの**1. アブレーション**だけ実行し直します。
- `scale=(0.2, 1.0)` は、元の写真の**最小20%まで**切り取る強い設定。
- **見るポイント**: `crop` の成績は上がる？ 下がる？ 強すぎると、被写体が写らない写真ができて逆効果にならないか？
- ⚠ 終わったら `(0.6, 1.0)` に**戻してください**。

### 演習2: 垂直反転を追加してみる（コードを足す練習）
`src/train.py` の `AUG_SETS` の中の `"flip"` の下に、次の1行を足します（コピーして貼り付け）。
```python
    "vflip": [v2.Resize((SIZE, SIZE)), v2.RandomVerticalFlip()],
```
そして、このNotebookの **`for aug in ["none", "flip", "crop", "color", "basic"]:`** の **`"basic"` の後ろに `, "vflip"`** を足して実行します。
- **見るポイント**: 上下反転は効く？ アリ・ハチは**上下逆さまの姿**で現れることがある？（Notebook 01 の考え方で判断）
- **分かること**: 「効く・効かない」は**課題の性質で決まる**。試して確かめるのが近道。

### 演習3: 色が重要な課題を考える（コード不要）
もし、分類する対象が**「熟したバナナ」と「未熟なバナナ」**だったら、**色調変更**は使うべきでしょうか？
- ヒント: 熟したかどうかを、AIは**何で**見分ける？ 色を変えたらラベルは変わってしまわないか？
- **分かること**: 拡張は「ラベルを変えない」範囲で使う。何がラベルの手がかりかを考えて選ぶ。

---

## ✅ まとめ

- 拡張は**1種類ずつ**試して効果を切り分ける。無駄な拡張（今回の色調）や、逆に**特に効く拡張**（今回の切り抜き）が見つかる。
- **何が効くかは課題によって違う**。アリ/ハチは「形」が手がかりなので、位置・大きさを変える拡張が効いた。
- 全部入りが最良とは限らない。**効くものに絞る**。
- 拡張は過学習（gap）を縮めるが、**写真が極端に少ない**とtest精度は上がらない。**拡張はデータの代わりにならない。**

➡ 次は **`05_advanced.ipynb`**（最終回）。拡張の選び方をおまかせにする方法、画像を混ぜる方法、そして**転移学習**を試します。